# Notebook 03: Seeded Sheet Fragmentation and Plasmoid Proxy

This notebook extends the current-sheet model by adding **sheet-localized multimode perturbations**.

Notebook 02 produced:

- one smooth shear layer
- one failed-threshold band
- one family of 45° contours

Notebook 03 asks what happens when that layer is seeded with structured perturbations along the sheet.

Core goal:

- turn one continuous failed band into **corrugated** or **pocketed** threshold structure
- keep the same cross-sheet cosine observable
- quantify fragmentation with simple, transparent proxy metrics

We keep the same geometric gate

\[
\cos\theta \ge \frac{1}{\sqrt{1^2 + 1^2}}
\]

and interpret the below-threshold region as a **strong-shear layer**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque

np.random.seed(11)

THRESHOLD = 1 / np.sqrt(2)

nx, ny = 360, 280
x = np.linspace(-6.0, 6.0, nx)
y = np.linspace(-3.0, 3.0, ny)
X, Y = np.meshgrid(x, y)

print("45° threshold =", THRESHOLD)
print("grid shape =", X.shape)

## 1. Field model and helpers

We retain the Harris-sheet-style base reversal

\[
B_x(y)=\tanh(y/\delta)
\]

and replace the earlier simple transverse perturbation with a sheet-localized multimode perturbation:

\[
B_y(x,y)=e^{-y^2/\sigma^2}\sum_n a_n \sin(k_n x + \phi_n).
\]

This is still not a full MHD solver. It is a geometric fragmentation model.

In [ ]:
def normalize_field(Bx, By):
    mag = np.sqrt(Bx**2 + By**2)
    mag = np.where(mag == 0, 1.0, mag)
    return Bx / mag, By / mag

def multimode_sheet_perturbation(X, Y, amps, ks, phases, sigma=0.38):
    out = np.zeros_like(X)
    envelope = np.exp(-(Y**2) / (sigma**2))
    for a, k, p in zip(amps, ks, phases):
        out += a * np.sin(k * X + p)
    return envelope * out

def magnetic_field_fragmented(X, Y, delta=0.35, base_eps=0.06, frag_amp=0.20, sigma=0.38):
    # Base reversal
    Bx = np.tanh(Y / delta)

    # Small smooth background perturbation
    By_background = base_eps * np.sin(1.2 * X) * np.exp(-(Y**2) / (1.0**2))

    # Multimode sheet-localized perturbation
    ks = [1.0, 2.0, 3.6, 5.0]
    phases = [0.0, 0.8, 1.7, 2.6]
    amps = [frag_amp, 0.75 * frag_amp, 0.55 * frag_amp, 0.35 * frag_amp]
    By_fragment = multimode_sheet_perturbation(X, Y, amps, ks, phases, sigma=sigma)

    By = By_background + By_fragment
    return Bx, By

def cross_sheet_cosine(Bx_hat, By_hat, shift=3):
    Bx_up = np.roll(Bx_hat, -shift, axis=0)
    By_up = np.roll(By_hat, -shift, axis=0)

    Bx_dn = np.roll(Bx_hat, shift, axis=0)
    By_dn = np.roll(By_hat, shift, axis=0)

    cos_map = Bx_up * Bx_dn + By_up * By_dn
    cos_map[:shift, :] = np.nan
    cos_map[-shift:, :] = np.nan
    return cos_map

def threshold_mask(cos_map, threshold=THRESHOLD):
    return cos_map < threshold

def estimate_failed_width_vs_x(cos_map, y, threshold=THRESHOLD):
    widths = np.full(cos_map.shape[1], np.nan)
    for j in range(cos_map.shape[1]):
        col = cos_map[:, j]
        bad = np.isfinite(col) & (col < threshold)
        if np.any(bad):
            y_bad = y[bad]
            widths[j] = y_bad.max() - y_bad.min()
    return widths

def connected_components(mask):
    visited = np.zeros_like(mask, dtype=bool)
    rows, cols = mask.shape
    count = 0
    sizes = []

    for i in range(rows):
        for j in range(cols):
            if mask[i, j] and not visited[i, j]:
                count += 1
                q = deque([(i, j)])
                visited[i, j] = True
                size = 0

                while q:
                    r, c = q.popleft()
                    size += 1
                    for rr, cc in [(r-1, c), (r+1, c), (r, c-1), (r, c+1)]:
                        if 0 <= rr < rows and 0 <= cc < cols:
                            if mask[rr, cc] and not visited[rr, cc]:
                                visited[rr, cc] = True
                                q.append((rr, cc))
                sizes.append(size)
    return count, sizes

def central_band_mask(mask, Y, half_width=0.8):
    return mask & (np.abs(Y) < half_width)

def local_min_count_1d(arr):
    arr = np.asarray(arr, dtype=float)
    good = np.isfinite(arr)
    idx = np.where(good)[0]
    if len(idx) < 3:
        return 0
    vals = arr[idx]
    count = 0
    for k in range(1, len(vals) - 1):
        if vals[k] < vals[k - 1] and vals[k] < vals[k + 1]:
            count += 1
    return count

## 2. Regimes to compare

We sweep a fragmentation amplitude parameter. This controls how strongly the sheet is corrugated along \(x\).

- weak: nearly one smooth band
- medium: visible corrugation
- strong: broken-up pocket structure

In [ ]:
delta = 0.35
shift = 3
frag_amps = [0.05, 0.14, 0.28, 0.42]

print("delta =", delta)
print("fragmentation amplitudes =", frag_amps)

## 3. Example vector field

A quick field plot for one intermediate regime.

In [ ]:
frag_amp_example = 0.28
Bx_ex, By_ex = magnetic_field_fragmented(X, Y, delta=delta, base_eps=0.06, frag_amp=frag_amp_example, sigma=0.38)
Bx_hat_ex, By_hat_ex = normalize_field(Bx_ex, By_ex)

skip = 12
plt.figure(figsize=(8, 5))
plt.quiver(X[::skip, ::skip], Y[::skip, ::skip], Bx_hat_ex[::skip, ::skip], By_hat_ex[::skip, ::skip])
plt.xlabel("x")
plt.ylabel("y")
plt.title(f"Normalized Magnetic Field (fragmentation amplitude={frag_amp_example})")
plt.show()

## 4. Run the fragmentation sweep

For each perturbation level we compute:

- cross-sheet cosine map
- failed-threshold mask
- 45° contour
- disconnected pocket count in the central sheet band
- width variation along \(x\)
- minimum cosine along the sheet center
- local-minimum count along the sheet center

In [ ]:
results = []

for frag_amp in frag_amps:
    Bx, By = magnetic_field_fragmented(X, Y, delta=delta, base_eps=0.06, frag_amp=frag_amp, sigma=0.38)
    Bx_hat, By_hat = normalize_field(Bx, By)

    cos_map = cross_sheet_cosine(Bx_hat, By_hat, shift=shift)
    failed_mask = threshold_mask(cos_map, threshold=THRESHOLD)

    central_failed = central_band_mask(failed_mask, Y, half_width=0.8)
    component_count, component_sizes = connected_components(np.nan_to_num(central_failed, nan=False).astype(bool))

    widths = estimate_failed_width_vs_x(cos_map, y, threshold=THRESHOLD)
    width_mean = float(np.nanmean(widths)) if np.any(np.isfinite(widths)) else np.nan
    width_std = float(np.nanstd(widths)) if np.any(np.isfinite(widths)) else np.nan

    center_row = np.argmin(np.abs(y - 0.0))
    centerline = cos_map[center_row, :]
    centerline_min = float(np.nanmin(centerline))
    centerline_min_count = int(local_min_count_1d(centerline))

    failed_fraction = float(np.nanmean(failed_mask))
    central_failed_fraction = float(np.nanmean(central_failed))

    results.append({
        "frag_amp": float(frag_amp),
        "Bx_hat": Bx_hat,
        "By_hat": By_hat,
        "cos_map": cos_map,
        "failed_mask": failed_mask,
        "central_failed": central_failed,
        "component_count": int(component_count),
        "component_sizes": component_sizes,
        "widths": widths,
        "width_mean": width_mean,
        "width_std": width_std,
        "centerline": centerline,
        "centerline_min": centerline_min,
        "centerline_min_count": centerline_min_count,
        "failed_fraction": failed_fraction,
        "central_failed_fraction": central_failed_fraction,
    })

summary = [
    {
        "frag_amp": r["frag_amp"],
        "failed_fraction": r["failed_fraction"],
        "central_failed_fraction": r["central_failed_fraction"],
        "component_count": r["component_count"],
        "width_mean": r["width_mean"],
        "width_std": r["width_std"],
        "centerline_min": r["centerline_min"],
        "centerline_min_count": r["centerline_min_count"],
    }
    for r in results
]

summary

## 5. Cross-sheet cosine maps with 45° contours

The white dashed line is the explicit 45° boundary.

This is the main geometry figure for the notebook.

In [ ]:
for r in results:
    plt.figure(figsize=(9, 4.8))
    plt.imshow(
        r["cos_map"],
        extent=[x.min(), x.max(), y.min(), y.max()],
        origin="lower",
        aspect="auto"
    )
    plt.colorbar(label="cross-sheet cosine")
    plt.contour(
        X, Y, r["cos_map"],
        levels=[THRESHOLD],
        colors="white",
        linewidths=1.8,
        linestyles="--"
    )
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(f"Cross-Sheet Cosine Map (fragmentation amplitude={r['frag_amp']})")
    plt.show()

## 6. Failed-threshold masks

These panels isolate the region where the cross-sheet cosine fails the 45° gate.

In [ ]:
for r in results:
    plt.figure(figsize=(9, 4.8))
    plt.imshow(
        r["failed_mask"],
        extent=[x.min(), x.max(), y.min(), y.max()],
        origin="lower",
        aspect="auto"
    )
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(f"Below-Threshold Region (fragmentation amplitude={r['frag_amp']})")
    plt.show()

## 7. Central-sheet pocket masks

To avoid counting far-away numerical artifacts, we also inspect a central \(|y|<0.8\) band only.

This is the most directly interpretable fragmentation mask in the notebook.

In [ ]:
for r in results:
    plt.figure(figsize=(9, 4.8))
    plt.imshow(
        r["central_failed"],
        extent=[x.min(), x.max(), y.min(), y.max()],
        origin="lower",
        aspect="auto"
    )
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(f"Central Failed Region (fragmentation amplitude={r['frag_amp']})")
    plt.show()

## 8. Failed-band width versus x

A smooth sheet gives an almost uniform width. A fragmented sheet develops stronger width variation along \(x\).

In [ ]:
for r in results:
    plt.figure(figsize=(9, 4))
    plt.plot(x, r["widths"])
    plt.xlabel("x")
    plt.ylabel("failed-band width in y")
    plt.title(f"Failed-Band Width vs x (fragmentation amplitude={r['frag_amp']})")
    plt.show()

## 9. Centerline cosine along the sheet

We sample the cosine map along the central row near \(y=0\). Pocket formation tends to create deeper and more numerous minima along this line.

In [ ]:
for r in results:
    plt.figure(figsize=(9, 4))
    plt.plot(x, r["centerline"])
    plt.axhline(THRESHOLD, linestyle="--")
    plt.xlabel("x")
    plt.ylabel("cross-sheet cosine at y≈0")
    plt.title(f"Centerline Cosine (fragmentation amplitude={r['frag_amp']})")
    plt.show()

## 10. Summary curves

These are the compact fragmentation metrics for the notebook.

In [ ]:
frag_vals = np.array([r["frag_amp"] for r in results])
failed_vals = np.array([r["failed_fraction"] for r in results])
central_failed_vals = np.array([r["central_failed_fraction"] for r in results])
component_vals = np.array([r["component_count"] for r in results])
width_mean_vals = np.array([r["width_mean"] for r in results])
width_std_vals = np.array([r["width_std"] for r in results])
centerline_min_vals = np.array([r["centerline_min"] for r in results])
centerline_min_count_vals = np.array([r["centerline_min_count"] for r in results])

plt.figure(figsize=(8, 5))
plt.plot(frag_vals, component_vals, marker="o")
plt.xlabel("fragmentation amplitude")
plt.ylabel("central component count")
plt.title("Pocket Count vs Fragmentation Amplitude")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(frag_vals, width_std_vals, marker="o")
plt.xlabel("fragmentation amplitude")
plt.ylabel("width std along x")
plt.title("Width Variation vs Fragmentation Amplitude")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(frag_vals, centerline_min_vals, marker="o")
plt.xlabel("fragmentation amplitude")
plt.ylabel("minimum centerline cosine")
plt.title("Deepest Centerline Shear vs Fragmentation Amplitude")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(frag_vals, centerline_min_count_vals, marker="o")
plt.xlabel("fragmentation amplitude")
plt.ylabel("local minima count along centerline")
plt.title("Centerline Pocket Count Proxy")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(frag_vals, failed_vals, marker="o", label="domain-wide below-threshold fraction")
plt.plot(frag_vals, central_failed_vals, marker="s", label="central-band below-threshold fraction")
plt.xlabel("fragmentation amplitude")
plt.ylabel("below-threshold fraction")
plt.title("Below-Threshold Fraction vs Fragmentation Amplitude")
plt.legend()
plt.show()

## 11. Compact printed summary

In [ ]:
for r in summary:
    print(
        f"frag_amp={r['frag_amp']:.2f} | "
        f"failed_fraction={r['failed_fraction']:.4f} | "
        f"central_failed_fraction={r['central_failed_fraction']:.4f} | "
        f"component_count={r['component_count']} | "
        f"width_mean={r['width_mean']:.4f} | "
        f"width_std={r['width_std']:.4f} | "
        f"centerline_min={r['centerline_min']:.4f} | "
        f"centerline_min_count={r['centerline_min_count']}"
    )

## 12. Interpretation

This notebook supports a more specific statement than Notebook 02:

> A smooth current-sheet shear layer can be driven into corrugated and pocketed threshold structure by sheet-localized multimode perturbations.

What the notebook does **not** claim:

- it does not solve full reconnection dynamics
- it does not evolve resistive MHD in time
- it does not prove plasmoid instability in the formal sense

What it **does** provide:

- a controlled geometric fragmentation model
- an explicit 45° boundary on the cross-sheet cosine field
- simple proxy metrics for pocket count, width variation, and centerline structure

That makes it a reasonable precursor to a next notebook on time evolution or stochastic/turbulent perturbation statistics.